# Lectura de Datos

In [1]:
import scipy.io 
import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.svm import SVC 
from auto_tqdm import tqdm
from sklearn.feature_selection import RFE
import pathlib
import typing
from typing import List, Tuple, Dict, Any, Callable, Type
import numpy.typing as npt
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator
from collections import Counter
Scaler = Type[BaseEstimator]
from datareader import *
from SAE import *

from featureselectors import RFE 

# Creating the algorithm of RFE using SAE for evaluation




# CUB - Train Val 

In [2]:
#################
### Arguments ###
#################
class Arguments:
    def __init__(self, dataset,seed,mode):
        self.dataset = dataset
        self.seed = seed
        self.mode = mode
        
         
DECIMAL_NUMBERS        = 3
FOLDS                  = 5
scaler_str             = 'Standard'     # ['Standard','MinMax']
orig_attribute         = False # Attributos originales u otros
debug                  = True 
SHOW                   = True

step                     = 5  # Paso de atributos a seleccionar [a, a+step, a+2*step, ...]
hitk = 1    
args = Arguments('CUB',42,'rfe') # MODE = RFE


#################
##### SetUp #####
#################
random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True

# Set up the path to the data
# Set up the path to the data
path_dir                  = pathlib.Path('data/')
path_data                 = path_dir / args.dataset  
results_dir               = pathlib.Path(f'Results/HAIS_v2/{args.dataset}') 


path_results_rfe_train_val    = results_dir / f'Wrapper{args.dataset}-RFE-train-val.csv'
path_results_random_val       = results_dir / f'Wrapper{args.dataset}-Random-val.csv' 
path_results_rfe_val_val      = results_dir / f'Wrapper{args.dataset}-RFE-val-val.csv'
path_results_random_simple    = results_dir / f'Wrapper{args.dataset}-Random-simple.csv' 

path_masks_rfe_train_val    = results_dir / f'Masks_{args.dataset}-RFE-train-val.csv'
path_masks_random_val       = results_dir / f'Masks_{args.dataset}-Random-val.csv' 
path_masks_rfe_val_val      = results_dir / f'Masks_{args.dataset}-RFE-val-val.csv'
path_masks_random_simple    = results_dir / f'Masks_{args.dataset}-Random-simple.csv' 

path_oracle_rfe_train_val    = results_dir / f'Oracle_Wrapper_{args.dataset}-RFE-train-val.csv'
path_oracle_random_val       = results_dir / f'Oracle_Wrapper_{args.dataset}-Random-val.csv' 
path_oracle_rfe_val_val      = results_dir / f'Oracle_Wrapper_{args.dataset}-RFE-val-val.csv'
path_oracle_random_simple    = results_dir / f'Oracle_Wrapper_{args.dataset}-Random-simple.csv' 

path_conteo_rfe_train_val    = results_dir / f'Conteo_{args.dataset}-RFE-train-val.csv'
path_conteo_random_val       = results_dir / f'Conteo_{args.dataset}-Random-val.csv' 
path_conteo_rfe_val_val      = results_dir / f'Conteo_{args.dataset}-RFE-val-val.csv'
path_conteo_random_simple    = results_dir / f'Conteo_{args.dataset}-Random-simple.csv' 



# Load the data
matcontent, att_splits = load_data_from_path(path_data, show = False)

"""
Comentar que en los datos de este datasets viene una partición establecida para train y val. Sin embargo, la partición de 'val' no contempla clases Unseen. Por tanto no la vamos usar.
Crearemos una partición de validación para clases Seen y Unseen a partir de la partición de test. 
Como conjunto de entrenamiento usaremos el que viene dado por los índices de la variable 'trainval_loc' en att_splits. 
Los índices de las varaibles 'train_loc' y 'val_loc' no los vamos a usar.
"""
# Get the data from att_splits dictionary
allclasses_names            = att_splits['allclasses_names']
original_att                = att_splits['original_att'].T.astype('float')  # Attributos originales
attribute                   = att_splits['att'].T.astype('float')           # Atributos que no se de donde salen, pero son los que usan <=== IMPORTANTE
# Indixes of the instances in the train, test and validation sets
test_unseen_loc             = att_splits['test_unseen_loc'].squeeze() - 1
test_seen_loc               = att_splits['test_seen_loc'].squeeze() - 1
trainval_loc                = att_splits['trainval_loc'].squeeze() - 1      # En caso de no usar validación, aquí estara todo el entrenamiento
train_loc                   = att_splits['train_loc'].squeeze() - 1         # Esto es si usamos validacion, ver comentario arriba
val_loc                     = att_splits['val_loc'].squeeze() -1            # Esto es si usamos validacion, ver comentario arriba
# Get the dta from matcontent dictionary
feature =  matcontent['features'].T                       # Feature Matrix (Number of instances x Number of features)
labels  =  matcontent['labels'].astype(int).squeeze() - 1 # Ponemos que sea entero, eliminamos la dimension extra y con el '-1' movemos el rango de [1,200] a [0,199]

random.shuffle(trainval_loc)
# Es importante usar aquí la matrices feature y labels originales
trainval_feature = feature[trainval_loc]
trainval_labels  = labels[trainval_loc]
trainval_classes = np.unique(trainval_labels)
random.shuffle(trainval_classes)

classes_per_fold = int(trainval_classes.shape[0] / FOLDS)
acarreo          = trainval_classes.shape[0] % FOLDS


# Get attributes and preprocessing them
attribute = original_att if orig_attribute else attribute
attribute = scaler_data(get_scaler(scaler_str),attribute)




#################
##### FOLDS #####
#################
folds = []
for i in range(FOLDS):
    
    # Gather data for the fold
    start = i * classes_per_fold
    end   = (i + 1) * classes_per_fold if i != FOLDS - 1 else (i + 1) * classes_per_fold + acarreo
    
    fold_i_val_classes       = trainval_classes[start:end] # Unseen
    fold_i_training_classes  = [x for x in trainval_classes if x not in fold_i_val_classes] # Seen
    
    fold_i_val_loc           = np.where(np.isin(trainval_labels, fold_i_val_classes)) # index for val
    fold_i_training_loc      = np.where(np.isin(trainval_labels, fold_i_training_classes)) # Index for training
    # np.isin(element,test_element) Calculates element in test_elements, broadcasting over element only. 
    # #Returns a boolean array of the same shape as element that is True where an element of element is in 
    # #test_elements and False otherwise.
    # np.where(lists[bool]) returns the indexes of the True values in the lists
    
    assert np.intersect1d(fold_i_val_loc, fold_i_training_loc).shape[0] == 0, f"Error: Problems found in creating folds {i}. Val and training classes are not disjoint"
    assert np.intersect1d(fold_i_val_classes, fold_i_training_classes).shape[0] == 0, f"Error: Problems found in creating folds {i}. Val and training classes are not disjoint"
    assert all([i not in fold_i_training_classes for i in trainval_labels[fold_i_val_loc]]), f"Error: Problems found in creating folds {i}"
    assert all([i not in fold_i_val_classes for i in trainval_labels[fold_i_training_loc]]), f"Error: Problems found in creating folds {i}"

    fold_i_training_features = trainval_feature[fold_i_training_loc]
    fold_i_val_features      = trainval_feature[fold_i_val_loc]
    fold_i_training_labels   = trainval_labels[fold_i_training_loc]
    fold_i_val_labels        = trainval_labels[fold_i_val_loc]
    fold_i_ss_val            = gen_ss_from_data(fold_i_val_labels,attribute)
    fold_i_ss_training       = gen_ss_from_data(fold_i_training_labels,attribute)
    
    assert np.intersect1d(np.unique(fold_i_val_labels),np.unique(fold_i_training_labels)).shape[0] == 0, f"Los conjuntos del fold {i+1} de etiquetas no son disjuntos"
    
    # Preprocess the data
    fold_i_training_features, fold_i_val_features = scaler_data(get_scaler(scaler_str),
                                                                fold_i_training_features,
                                                                fold_i_val_features)
    
    # Creating Datasets 
    fold_training = Dataset(features = fold_i_training_features, 
                            labels   = fold_i_training_labels, 
                            att      = fold_i_ss_training, 
                            mode     = 'train')
    
    fold_val      = Dataset(features = fold_i_val_features,
                            labels   = fold_i_val_labels,
                            att      = fold_i_ss_val,
                            mode     = 'val')
    
    fold = StackedDataset({'train':fold_training, 'val':fold_val})
    
    folds.append(fold)
    
    if SHOW:
        print(f"Fold {i+1} || Val Classes {np.unique(fold_i_val_labels).shape} - {fold_i_val_labels.shape} instances || Training Classes {np.unique(fold_i_training_labels).shape} - {fold_i_training_labels.shape} instances")
        
        
        







args = Arguments('CUB',42,'rfe') # MODE = RFE
random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True

vattributes = np.arange(10, attribute.shape[1], step)
if attribute.shape[1] % step != 0:
    vattributes = np.append(vattributes, attribute.shape[1])

conteo          = np.zeros(attribute.shape[1])
best_folds_configuration = [] # Mejor configurración de atributos seleccionados en cada uno de los folds
for i, fold in enumerate(folds):

    print(f"Fold {i+1}...")

    training_data = fold.datasets['train']
    val_data      = fold.datasets['val']

    attributes_X_train = training_data.att.copy()

    rfe = RFE(estimator = None, estimator_name = 'SAE', n_features_to_select = 1, step = 1, verbose = True)
    rfe.fit(training_data.features, training_data.att,training_data.labels)
    ranking = rfe.get_ranking()
    
    fold_i_att_acc = {'Attributes': [],
                      'Train Acc': [],
                      'Val Acc': [],
                      'Mask':[]}
    
    for nv in vattributes:
        
        mask = ranking <= nv
        W = SAE(training_data.features.T, training_data.att[:,mask].T,500000) 
        # Training Acc
        gt_ss_training_adapted  = attribute[training_data.classes][:,mask]
        semantic_predicted = np.dot(training_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, training_data.classes,training_data.labels)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        # Val Acc
        gt_ss_val_adapted    = attribute[val_data.classes][:,mask]
        semantic_predicted = np.dot(val_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, val_data.classes,val_data.labels)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        # Save results
        fold_i_att_acc['Attributes'].append(nv)
        fold_i_att_acc['Train Acc'].append(zsl_accuracy_train)
        fold_i_att_acc['Val Acc'].append(zsl_accuracy1_unseen)
        fold_i_att_acc['Mask'].append(mask)
        
        if debug:
            print(f"nv: {nv} || Train: {zsl_accuracy_train} || Val: {zsl_accuracy1_unseen}")
    # Saving the best configuration of the fold
    fold_i_att_acc = pd.DataFrame(fold_i_att_acc).sort_values(by = 'Val Acc', ascending = False)
    best_folds_configuration.append(fold_i_att_acc.iloc[0])
    

for res in best_folds_configuration:
    conteo = conteo + np.asarray(res['Mask']).astype(int)
    
    
df = {}
for i in range(len(best_folds_configuration)):
    df[i] = best_folds_configuration[i].to_dict()
df = pd.DataFrame(df).T

if rfe_mode: 
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_rfe_train_val)
else:
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_random_val)
    
    
    
    
# Evaluamos en test 
test_results = {'Counts':[],'Attributes':[],'Train Acc':[],'Test Seen Acc':[],'Test Unseen Acc':[]}
j = 1
cadena = '5'
for i in range(FOLDS*2-1):
    if i < FOLDS:
        test_results['Counts'].append(str(FOLDS-i))
        test_results['Attributes'].append(np.where(conteo==FOLDS-i)[0].shape)
        mejores = np.where(conteo==FOLDS-i)[0]

    else:
        cadena = cadena+'-'+str(FOLDS-j)
        test_results['Counts'].append(cadena)
        test_results['Attributes'].append(np.where(conteo>=FOLDS-j)[0].shape)
        mejores = np.where(conteo>=FOLDS-j)[0]
        j+=1

    
    
    if mejores.shape[0] > 0:
        # Training
        training_feature = feature[trainval_loc]
        training_label   = labels[trainval_loc]
        ss_training      = gen_ss_from_data(training_label,attribute)
        # Get test
        test_unseen_feature = feature[test_unseen_loc]
        test_unseen_label   = labels[test_unseen_loc]
        ss_test_unseen      = gen_ss_from_data(test_unseen_label,attribute)
        test_seen_feature = feature[test_seen_loc]
        test_seen_label   = labels[test_seen_loc]
        ss_test_seen      = gen_ss_from_data(test_seen_label,attribute)
        # Preprocessing
        scaler = get_scaler(scaler_str)
        scaler.fit(training_feature)
        training_feature = scaler.transform(training_feature)
        test_unseen_feature = scaler.transform(test_unseen_feature)
        test_seen_feature = scaler.transform(test_seen_feature)
        # Fit the model
        W = SAE(training_feature.T, ss_training[:,mejores].T,500000) 
        gt_ss_training_adapted  = attribute[np.unique(training_label)][:,mejores]
        semantic_predicted = np.dot(training_feature, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, np.unique(training_label),training_label)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        test_results['Train Acc'].append(zsl_accuracy_train)
        # Evaluo en test seen
        gt_ss_val_adapted    = attribute[np.unique(test_seen_label)][:,mejores]
        semantic_predicted = np.dot(test_seen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_seen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_seen_label),test_seen_label)
        zsl_accuracy1_seen = round(zsl_accuracy1_seen,DECIMAL_NUMBERS)
        test_results['Test Seen Acc'].append(zsl_accuracy1_seen)
        # Evaluo en test unseen
        gt_ss_val_adapted    = attribute[np.unique(test_unseen_label)][:,mejores]
        semantic_predicted = np.dot(test_unseen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_unseen_label),test_unseen_label)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        test_results['Test Unseen Acc'].append(zsl_accuracy1_unseen)
    else:
        test_results['Train Acc'].append(0)
        test_results['Test Seen Acc'].append(0)
        test_results['Test Unseen Acc'].append(0)
    
if rfe_mode:
    pd.DataFrame(test_results).to_csv(path_oracle_rfe_train_val)
else:
    pd.DataFrame(test_results).to_csv(path_oracle_random_val)
    
pd.DataFrame(test_results)
        

You've selected CUB dataset
You've selected rfe mode
Fold 1 || Val Classes (30,) - (1400,) instances || Training Classes (120,) - (5657,) instances
Fold 2 || Val Classes (30,) - (1465,) instances || Training Classes (120,) - (5592,) instances
Fold 3 || Val Classes (30,) - (1386,) instances || Training Classes (120,) - (5671,) instances
Fold 4 || Val Classes (30,) - (1406,) instances || Training Classes (120,) - (5651,) instances
Fold 5 || Val Classes (30,) - (1400,) instances || Training Classes (120,) - (5657,) instances
You've selected CUB dataset
You've selected rfe mode
Fold 1...
Ranking: [79, 17, 39, 167, 105, 197, 120, 24, 112, 9, 58, 248, 285, 32, 66, 256, 207, 115, 268, 118, 235, 273, 286, 52, 210, 49, 175, 102, 8, 278, 233, 282, 31, 87, 95, 77, 182, 261, 133, 185, 296, 127, 18, 16, 155, 89, 204, 65, 106, 172, 293, 161, 33, 6, 258, 152, 290, 114, 291, 222, 276, 302, 287, 40, 190, 164, 198, 148, 205, 121, 275, 47, 110, 100, 22, 232, 70, 231, 265, 186, 46, 109, 220, 272, 260, 267

,Counts,Attributes,Train Acc,Test Seen Acc,Test Unseen Acc
0,5,"(164,)",91.937,57.993,41.052
1,4,"(130,)",91.668,57.426,36.131
2,3,"(15,)",61.046,26.474,19.077
3,2,"(3,)",6.334,3.231,4.988
4,1,"(0,)",0.000,0.000,0.000
5,5-4,"(294,)",93.297,60.658,41.658
6,5-4-3,"(309,)",93.439,60.998,41.827
7,5-4-3-2,"(312,)",93.411,60.941,41.692
8,5-4-3-2-1,"(312,)",93.411,60.941,41.692


# CUB - Val Val

In [3]:
args = Arguments('CUB',42,'rfe') # MODE = RFE


random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True




vattributes = np.arange(10, attribute.shape[1], step)
if attribute.shape[1] % step != 0:
    vattributes = np.append(vattributes, attribute.shape[1])

conteo          = np.zeros(attribute.shape[1])
best_folds_configuration = [] # Mejor configurración de atributos seleccionados en cada uno de los folds
for i, fold in enumerate(folds):

    print(f"Fold {i+1}...")

    training_data = fold.datasets['train']
    val_data      = fold.datasets['val']



    rfe = RFE(estimator = None, estimator_name = 'SAE', n_features_to_select = 1, step = 1, verbose = True)
    rfe.fit(val_data.features, val_data.att,val_data.labels)
    ranking = rfe.get_ranking()
    
    fold_i_att_acc = {'Attributes': [],
                      'Train Acc': [],
                      'Val Acc': [],
                      'Mask':[]}
    
    for nv in vattributes:
        
        mask = ranking <= nv
        W = SAE(training_data.features.T, training_data.att[:,mask].T,500000) 
        # Training Acc
        gt_ss_training_adapted  = attribute[training_data.classes][:,mask]
        semantic_predicted = np.dot(training_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, training_data.classes,training_data.labels)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        # Val Acc
        gt_ss_val_adapted    = attribute[val_data.classes][:,mask]
        semantic_predicted = np.dot(val_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, val_data.classes,val_data.labels)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        # Save results
        fold_i_att_acc['Attributes'].append(nv)
        fold_i_att_acc['Train Acc'].append(zsl_accuracy_train)
        fold_i_att_acc['Val Acc'].append(zsl_accuracy1_unseen)
        fold_i_att_acc['Mask'].append(mask)
        
        if debug:
            print(f"nv: {nv} || Train: {zsl_accuracy_train} || Val: {zsl_accuracy1_unseen}")
    # Saving the best configuration of the fold
    fold_i_att_acc = pd.DataFrame(fold_i_att_acc).sort_values(by = 'Val Acc', ascending = False)
    best_folds_configuration.append(fold_i_att_acc.iloc[0])
    

for res in best_folds_configuration:
    conteo = conteo + np.asarray(res['Mask']).astype(int)
   
   
   
   
df = {}
for i in range(len(best_folds_configuration)):
    df[i] = best_folds_configuration[i].to_dict()
df = pd.DataFrame(df).T

if rfe_mode: # Los mejores resultados de cada fold y el conteo
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_rfe_val_val)
else:
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_random_val)
    
    
    
    
    
    
    
    
# Evaluamos en test 
test_results = {'Counts':[],'Attributes':[],'Train Acc':[],'Test Seen Acc':[],'Test Unseen Acc':[]}
j = 1
cadena = '5'
for i in range(FOLDS*2-1):
    if i < FOLDS:
        test_results['Counts'].append(str(FOLDS-i))
        test_results['Attributes'].append(np.where(conteo==FOLDS-i)[0].shape)
        mejores = np.where(conteo==FOLDS-i)[0]
        print(FOLDS-i)
    else:
        cadena = cadena+'-'+str(FOLDS-j)
        test_results['Counts'].append(cadena)
        test_results['Attributes'].append(np.where(conteo>=FOLDS-j)[0].shape)
        mejores = np.where(conteo>=FOLDS-j)[0]
        j+=1
        print(FOLDS-j)
    
    
    if mejores.shape[0] > 0:
        # Training
        training_feature = feature[trainval_loc].copy()
        training_label   = labels[trainval_loc].copy()
        ss_training      = gen_ss_from_data(training_label,attribute)
        # Get test
        test_unseen_feature = feature[test_unseen_loc].copy()
        test_unseen_label   = labels[test_unseen_loc].copy()
        ss_test_unseen      = gen_ss_from_data(test_unseen_label,attribute)
        test_seen_feature = feature[test_seen_loc].copy()
        test_seen_label   = labels[test_seen_loc].copy()
        ss_test_seen      = gen_ss_from_data(test_seen_label,attribute)
        # Preprocessing
        scaler = get_scaler(scaler_str)
        scaler.fit(training_feature)
        training_feature = scaler.transform(training_feature)
        test_unseen_feature = scaler.transform(test_unseen_feature)
        test_seen_feature = scaler.transform(test_seen_feature)
        # Fit the model
        W = SAE(training_feature.T, ss_training[:,mejores].T,500000) 
        gt_ss_training_adapted  = attribute[np.unique(training_label)][:,mejores]
        semantic_predicted = np.dot(training_feature, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, np.unique(training_label),training_label)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        test_results['Train Acc'].append(zsl_accuracy_train)
        # Evaluo en test seen
        gt_ss_val_adapted    = attribute[np.unique(test_seen_label)][:,mejores]
        semantic_predicted = np.dot(test_seen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_seen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_seen_label),test_seen_label)
        zsl_accuracy1_seen = round(zsl_accuracy1_seen,DECIMAL_NUMBERS)
        test_results['Test Seen Acc'].append(zsl_accuracy1_seen)
        # Evaluo en test unseen
        gt_ss_val_adapted    = attribute[np.unique(test_unseen_label)][:,mejores]
        semantic_predicted = np.dot(test_unseen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_unseen_label),test_unseen_label)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        test_results['Test Unseen Acc'].append(zsl_accuracy1_unseen)
    else:
        test_results['Train Acc'].append(0)
        test_results['Test Seen Acc'].append(0)
        test_results['Test Unseen Acc'].append(0)
    
if rfe_mode:
    pd.DataFrame(test_results).to_csv(path_oracle_rfe_val_val)
else:
    pd.DataFrame(test_results).to_csv(path_oracle_random_val)
    
pd.DataFrame(test_results)

You've selected CUB dataset
You've selected rfe mode
Fold 1...
Ranking: [271, 276, 110, 292, 187, 154, 299, 104, 81, 28, 307, 60, 164, 117, 157, 64, 149, 21, 294, 111, 237, 304, 226, 199, 241, 166, 219, 5, 183, 124, 286, 75, 106, 74, 42, 91, 1, 168, 261, 54, 68, 217, 122, 162, 265, 71, 259, 210, 174, 290, 94, 17, 258, 9, 51, 53, 123, 280, 272, 240, 34, 99, 90, 115, 300, 218, 172, 119, 141, 215, 189, 188, 10, 182, 229, 105, 293, 242, 225, 305, 169, 57, 282, 88, 129, 55, 93, 11, 49, 233, 84, 48, 0, 22, 27, 216, 69, 36, 178, 52, 288, 58, 245, 65, 20, 281, 236, 262, 306, 244, 235, 238, 15, 186, 116, 152, 46, 311, 72, 206, 273, 62, 82, 243, 179, 107, 205, 274, 138, 297, 128, 254, 145, 267, 139, 14, 151, 185, 103, 228, 232, 194, 231, 153, 204, 50, 296, 192, 253, 213, 220, 184, 266, 112, 275, 4, 33, 101, 212, 100, 250, 89, 159, 279, 35, 291, 85, 37, 256, 308, 67, 222, 73, 180, 136, 135, 134, 303, 202, 175, 161, 264, 302, 120, 8, 41, 39, 310, 79, 283, 7, 25, 44, 97, 268, 121, 59, 227, 61, 234,

,Counts,Attributes,Train Acc,Test Seen Acc,Test Unseen Acc
0,5,"(245,)",93.184,60.261,41.523
1,4,"(65,)",86.552,47.902,33.771
2,3,"(2,)",3.259,2.098,5.460
3,2,"(0,)",0.000,0.000,0.000
4,1,"(0,)",0.000,0.000,0.000
5,5-4,"(310,)",93.368,61.054,41.793
6,5-4-3,"(312,)",93.411,60.941,41.692
7,5-4-3-2,"(312,)",93.411,60.941,41.692
8,5-4-3-2-1,"(312,)",93.411,60.941,41.692


# SUN

In [4]:
#################
### Arguments ###
#################
class Arguments:
    def __init__(self, dataset,seed,mode):
        self.dataset = dataset
        self.seed = seed
        self.mode = mode
        
         
DECIMAL_NUMBERS        = 3
FOLDS                  = 5
scaler_str             = 'Standard'     # ['Standard','MinMax']
orig_attribute         = False # Attributos originales u otros
debug                  = True 
SHOW                   = True

step                     = 5  # Paso de atributos a seleccionar [a, a+step, a+2*step, ...]
hitk = 1    
args = Arguments('SUN',42,'rfe') # MODE = RFE


#################
##### SetUp #####
#################
random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True

# Set up the path to the data
# Set up the path to the data
path_dir                  = pathlib.Path('data/')
path_data                 = path_dir / args.dataset  
results_dir               = pathlib.Path(f'Results/HAIS_v2/{args.dataset}') 


path_results_rfe_train_val    = results_dir / f'Wrapper{args.dataset}-RFE-train-val.csv'
path_results_random_val       = results_dir / f'Wrapper{args.dataset}-Random-val.csv' 
path_results_rfe_val_val      = results_dir / f'Wrapper{args.dataset}-RFE-val-val.csv'
path_results_random_simple    = results_dir / f'Wrapper{args.dataset}-Random-simple.csv' 

path_masks_rfe_train_val    = results_dir / f'Masks_{args.dataset}-RFE-train-val.csv'
path_masks_random_val       = results_dir / f'Masks_{args.dataset}-Random-val.csv' 
path_masks_rfe_val_val      = results_dir / f'Masks_{args.dataset}-RFE-val-val.csv'
path_masks_random_simple    = results_dir / f'Masks_{args.dataset}-Random-simple.csv' 

path_oracle_rfe_train_val    = results_dir / f'Oracle_Wrapper_{args.dataset}-RFE-train-val.csv'
path_oracle_random_val       = results_dir / f'Oracle_Wrapper_{args.dataset}-Random-val.csv' 
path_oracle_rfe_val_val      = results_dir / f'Oracle_Wrapper_{args.dataset}-RFE-val-val.csv'
path_oracle_random_simple    = results_dir / f'Oracle_Wrapper_{args.dataset}-Random-simple.csv' 

path_conteo_rfe_train_val    = results_dir / f'Conteo_{args.dataset}-RFE-train-val.csv'
path_conteo_random_val       = results_dir / f'Conteo_{args.dataset}-Random-val.csv' 
path_conteo_rfe_val_val      = results_dir / f'Conteo_{args.dataset}-RFE-val-val.csv'
path_conteo_random_simple    = results_dir / f'Conteo_{args.dataset}-Random-simple.csv' 



# Load the data
matcontent, att_splits = load_data_from_path(path_data, show = False)

"""
Comentar que en los datos de este datasets viene una partición establecida para train y val. Sin embargo, la partición de 'val' no contempla clases Unseen. Por tanto no la vamos usar.
Crearemos una partición de validación para clases Seen y Unseen a partir de la partición de test. 
Como conjunto de entrenamiento usaremos el que viene dado por los índices de la variable 'trainval_loc' en att_splits. 
Los índices de las varaibles 'train_loc' y 'val_loc' no los vamos a usar.
"""
# Get the data from att_splits dictionary
allclasses_names            = att_splits['allclasses_names']
original_att                = att_splits['original_att'].T.astype('float')  # Attributos originales
attribute                   = att_splits['att'].T.astype('float')           # Atributos que no se de donde salen, pero son los que usan <=== IMPORTANTE
# Indixes of the instances in the train, test and validation sets
test_unseen_loc             = att_splits['test_unseen_loc'].squeeze() - 1
test_seen_loc               = att_splits['test_seen_loc'].squeeze() - 1
trainval_loc                = att_splits['trainval_loc'].squeeze() - 1      # En caso de no usar validación, aquí estara todo el entrenamiento
train_loc                   = att_splits['train_loc'].squeeze() - 1         # Esto es si usamos validacion, ver comentario arriba
val_loc                     = att_splits['val_loc'].squeeze() -1            # Esto es si usamos validacion, ver comentario arriba
# Get the dta from matcontent dictionary
feature =  matcontent['features'].T                       # Feature Matrix (Number of instances x Number of features)
labels  =  matcontent['labels'].astype(int).squeeze() - 1 # Ponemos que sea entero, eliminamos la dimension extra y con el '-1' movemos el rango de [1,200] a [0,199]

random.shuffle(trainval_loc)
# Es importante usar aquí la matrices feature y labels originales
trainval_feature = feature[trainval_loc]
trainval_labels  = labels[trainval_loc]
trainval_classes = np.unique(trainval_labels)
random.shuffle(trainval_classes)

classes_per_fold = int(trainval_classes.shape[0] / FOLDS)
acarreo          = trainval_classes.shape[0] % FOLDS


# Get attributes and preprocessing them
attribute = original_att if orig_attribute else attribute
attribute = scaler_data(get_scaler(scaler_str),attribute)




#################
##### FOLDS #####
#################
folds = []
for i in range(FOLDS):
    
    # Gather data for the fold
    start = i * classes_per_fold
    end   = (i + 1) * classes_per_fold if i != FOLDS - 1 else (i + 1) * classes_per_fold + acarreo
    
    fold_i_val_classes       = trainval_classes[start:end] # Unseen
    fold_i_training_classes  = [x for x in trainval_classes if x not in fold_i_val_classes] # Seen
    
    fold_i_val_loc           = np.where(np.isin(trainval_labels, fold_i_val_classes)) # index for val
    fold_i_training_loc      = np.where(np.isin(trainval_labels, fold_i_training_classes)) # Index for training
    # np.isin(element,test_element) Calculates element in test_elements, broadcasting over element only. 
    # #Returns a boolean array of the same shape as element that is True where an element of element is in 
    # #test_elements and False otherwise.
    # np.where(lists[bool]) returns the indexes of the True values in the lists
    
    assert np.intersect1d(fold_i_val_loc, fold_i_training_loc).shape[0] == 0, f"Error: Problems found in creating folds {i}. Val and training classes are not disjoint"
    assert np.intersect1d(fold_i_val_classes, fold_i_training_classes).shape[0] == 0, f"Error: Problems found in creating folds {i}. Val and training classes are not disjoint"
    assert all([i not in fold_i_training_classes for i in trainval_labels[fold_i_val_loc]]), f"Error: Problems found in creating folds {i}"
    assert all([i not in fold_i_val_classes for i in trainval_labels[fold_i_training_loc]]), f"Error: Problems found in creating folds {i}"

    fold_i_training_features = trainval_feature[fold_i_training_loc]
    fold_i_val_features      = trainval_feature[fold_i_val_loc]
    fold_i_training_labels   = trainval_labels[fold_i_training_loc]
    fold_i_val_labels        = trainval_labels[fold_i_val_loc]
    fold_i_ss_val            = gen_ss_from_data(fold_i_val_labels,attribute)
    fold_i_ss_training       = gen_ss_from_data(fold_i_training_labels,attribute)
    
    assert np.intersect1d(np.unique(fold_i_val_labels),np.unique(fold_i_training_labels)).shape[0] == 0, f"Los conjuntos del fold {i+1} de etiquetas no son disjuntos"
    
    # Preprocess the data
    fold_i_training_features, fold_i_val_features = scaler_data(get_scaler(scaler_str),
                                                                fold_i_training_features,
                                                                fold_i_val_features)
    
    # Creating Datasets 
    fold_training = Dataset(features = fold_i_training_features, 
                            labels   = fold_i_training_labels, 
                            att      = fold_i_ss_training, 
                            mode     = 'train')
    
    fold_val      = Dataset(features = fold_i_val_features,
                            labels   = fold_i_val_labels,
                            att      = fold_i_ss_val,
                            mode     = 'val')
    
    fold = StackedDataset({'train':fold_training, 'val':fold_val})
    
    folds.append(fold)
    
    if SHOW:
        print(f"Fold {i+1} || Val Classes {np.unique(fold_i_val_labels).shape} - {fold_i_val_labels.shape} instances || Training Classes {np.unique(fold_i_training_labels).shape} - {fold_i_training_labels.shape} instances")
        
        

You've selected SUN dataset
You've selected rfe mode
Fold 1 || Val Classes (129,) - (2064,) instances || Training Classes (516,) - (8256,) instances
Fold 2 || Val Classes (129,) - (2064,) instances || Training Classes (516,) - (8256,) instances
Fold 3 || Val Classes (129,) - (2064,) instances || Training Classes (516,) - (8256,) instances
Fold 4 || Val Classes (129,) - (2064,) instances || Training Classes (516,) - (8256,) instances
Fold 5 || Val Classes (129,) - (2064,) instances || Training Classes (516,) - (8256,) instances


## Train Val

In [5]:
args = Arguments('SUN',42,'rfe') # MODE = RFE
random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True

vattributes = np.arange(10, attribute.shape[1], step)
if attribute.shape[1] % step != 0:
    vattributes = np.append(vattributes, attribute.shape[1])

conteo          = np.zeros(attribute.shape[1])
best_folds_configuration = [] # Mejor configurración de atributos seleccionados en cada uno de los folds
for i, fold in enumerate(folds):

    print(f"Fold {i+1}...")

    training_data = fold.datasets['train']
    val_data      = fold.datasets['val']

    attributes_X_train = training_data.att.copy()

    rfe = RFE(estimator = None, estimator_name = 'SAE', n_features_to_select = 1, step = 1, verbose = True)
    rfe.fit(training_data.features, training_data.att,training_data.labels)
    ranking = rfe.get_ranking()
    
    fold_i_att_acc = {'Attributes': [],
                      'Train Acc': [],
                      'Val Acc': [],
                      'Mask':[]}
    
    for nv in vattributes:
        
        mask = ranking <= nv
        W = SAE(training_data.features.T, training_data.att[:,mask].T,500000) 
        # Training Acc
        gt_ss_training_adapted  = attribute[training_data.classes][:,mask]
        semantic_predicted = np.dot(training_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, training_data.classes,training_data.labels)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        # Val Acc
        gt_ss_val_adapted    = attribute[val_data.classes][:,mask]
        semantic_predicted = np.dot(val_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, val_data.classes,val_data.labels)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        # Save results
        fold_i_att_acc['Attributes'].append(nv)
        fold_i_att_acc['Train Acc'].append(zsl_accuracy_train)
        fold_i_att_acc['Val Acc'].append(zsl_accuracy1_unseen)
        fold_i_att_acc['Mask'].append(mask)
        
        if debug:
            print(f"nv: {nv} || Train: {zsl_accuracy_train} || Val: {zsl_accuracy1_unseen}")
    # Saving the best configuration of the fold
    fold_i_att_acc = pd.DataFrame(fold_i_att_acc).sort_values(by = 'Val Acc', ascending = False)
    best_folds_configuration.append(fold_i_att_acc.iloc[0])
    

for res in best_folds_configuration:
    conteo = conteo + np.asarray(res['Mask']).astype(int)
    
    
df = {}
for i in range(len(best_folds_configuration)):
    df[i] = best_folds_configuration[i].to_dict()
df = pd.DataFrame(df).T

if rfe_mode: 
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_rfe_train_val)
else:
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_random_val)
    
    
    
    
# Evaluamos en test 
test_results = {'Counts':[],'Attributes':[],'Train Acc':[],'Test Seen Acc':[],'Test Unseen Acc':[]}
j = 1
cadena = '5'
for i in range(FOLDS*2-1):
    if i < FOLDS:
        test_results['Counts'].append(str(FOLDS-i))
        test_results['Attributes'].append(np.where(conteo==FOLDS-i)[0].shape)
        mejores = np.where(conteo==FOLDS-i)[0]

    else:
        cadena = cadena+'-'+str(FOLDS-j)
        test_results['Counts'].append(cadena)
        test_results['Attributes'].append(np.where(conteo>=FOLDS-j)[0].shape)
        mejores = np.where(conteo>=FOLDS-j)[0]
        j+=1

    
    
    if mejores.shape[0] > 0:
        # Training
        training_feature = feature[trainval_loc]
        training_label   = labels[trainval_loc]
        ss_training      = gen_ss_from_data(training_label,attribute)
        # Get test
        test_unseen_feature = feature[test_unseen_loc]
        test_unseen_label   = labels[test_unseen_loc]
        ss_test_unseen      = gen_ss_from_data(test_unseen_label,attribute)
        test_seen_feature = feature[test_seen_loc]
        test_seen_label   = labels[test_seen_loc]
        ss_test_seen      = gen_ss_from_data(test_seen_label,attribute)
        # Preprocessing
        scaler = get_scaler(scaler_str)
        scaler.fit(training_feature)
        training_feature = scaler.transform(training_feature)
        test_unseen_feature = scaler.transform(test_unseen_feature)
        test_seen_feature = scaler.transform(test_seen_feature)
        # Fit the model
        W = SAE(training_feature.T, ss_training[:,mejores].T,500000) 
        gt_ss_training_adapted  = attribute[np.unique(training_label)][:,mejores]
        semantic_predicted = np.dot(training_feature, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, np.unique(training_label),training_label)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        test_results['Train Acc'].append(zsl_accuracy_train)
        # Evaluo en test seen
        gt_ss_val_adapted    = attribute[np.unique(test_seen_label)][:,mejores]
        semantic_predicted = np.dot(test_seen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_seen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_seen_label),test_seen_label)
        zsl_accuracy1_seen = round(zsl_accuracy1_seen,DECIMAL_NUMBERS)
        test_results['Test Seen Acc'].append(zsl_accuracy1_seen)
        # Evaluo en test unseen
        gt_ss_val_adapted    = attribute[np.unique(test_unseen_label)][:,mejores]
        semantic_predicted = np.dot(test_unseen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_unseen_label),test_unseen_label)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        test_results['Test Unseen Acc'].append(zsl_accuracy1_unseen)
    else:
        test_results['Train Acc'].append(0)
        test_results['Test Seen Acc'].append(0)
        test_results['Test Unseen Acc'].append(0)
    
if rfe_mode:
    pd.DataFrame(test_results).to_csv(path_oracle_rfe_train_val)
else:
    pd.DataFrame(test_results).to_csv(path_oracle_random_val)
    
pd.DataFrame(test_results)

You've selected SUN dataset
You've selected rfe mode
Fold 1...
Ranking: [51, 58, 24, 63, 5, 26, 7, 12, 2, 43, 101, 60, 71, 55, 44, 18, 19, 59, 48, 100, 87, 97, 47, 99, 10, 98, 22, 50, 11, 52, 9, 95, 33, 29, 42, 45, 34, 39, 27, 65, 70, 17, 31, 16, 23, 82, 20, 37, 1, 89, 69, 73, 32, 68, 21, 46, 6, 57, 28, 78, 83, 41, 79, 84, 15, 96, 35, 93, 64, 74, 77, 91, 62, 38, 53, 25, 3, 56, 4, 14, 40, 85, 86, 94, 90, 72, 54, 30, 67, 36, 61, 49, 88, 66, 13, 92, 8, 80, 81, 76, 0, 75]
nv: 10 || Train: 20.422 || Val: 13.421
nv: 15 || Train: 30.85 || Val: 15.116
nv: 20 || Train: 38.215 || Val: 17.733
nv: 25 || Train: 43.593 || Val: 21.415
nv: 30 || Train: 49.31 || Val: 25.436
nv: 35 || Train: 52.289 || Val: 25.824
nv: 40 || Train: 53.646 || Val: 27.229
nv: 45 || Train: 57.207 || Val: 29.457
nv: 50 || Train: 58.442 || Val: 31.541
nv: 55 || Train: 59.193 || Val: 32.558
nv: 60 || Train: 61.301 || Val: 34.012
nv: 65 || Train: 63.59 || Val: 34.545
nv: 70 || Train: 64.971 || Val: 35.465
nv: 75 || Train: 67.297

,Counts,Attributes,Train Acc,Test Seen Acc,Test Unseen Acc
0,5,"(102,)",61.531,29.419,47.153
1,4,"(0,)",0.000,0.000,0.000
2,3,"(0,)",0.000,0.000,0.000
3,2,"(0,)",0.000,0.000,0.000
4,1,"(0,)",0.000,0.000,0.000
5,5-4,"(102,)",61.531,29.419,47.153
6,5-4-3,"(102,)",61.531,29.419,47.153
7,5-4-3-2,"(102,)",61.531,29.419,47.153
8,5-4-3-2-1,"(102,)",61.531,29.419,47.153


## Val Val

In [6]:
args = Arguments('SUN',42,'rfe') # MODE = RFE


random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True




vattributes = np.arange(10, attribute.shape[1], step)
if attribute.shape[1] % step != 0:
    vattributes = np.append(vattributes, attribute.shape[1])

conteo          = np.zeros(attribute.shape[1])
best_folds_configuration = [] # Mejor configurración de atributos seleccionados en cada uno de los folds
for i, fold in enumerate(folds):

    print(f"Fold {i+1}...")

    training_data = fold.datasets['train']
    val_data      = fold.datasets['val']



    rfe = RFE(estimator = None, estimator_name = 'SAE', n_features_to_select = 1, step = 1, verbose = True)
    rfe.fit(val_data.features, val_data.att,val_data.labels)
    ranking = rfe.get_ranking()
    
    fold_i_att_acc = {'Attributes': [],
                      'Train Acc': [],
                      'Val Acc': [],
                      'Mask':[]}
    
    for nv in vattributes:
        
        mask = ranking <= nv
        W = SAE(training_data.features.T, training_data.att[:,mask].T,500000) 
        # Training Acc
        gt_ss_training_adapted  = attribute[training_data.classes][:,mask]
        semantic_predicted = np.dot(training_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, training_data.classes,training_data.labels)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        # Val Acc
        gt_ss_val_adapted    = attribute[val_data.classes][:,mask]
        semantic_predicted = np.dot(val_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, val_data.classes,val_data.labels)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        # Save results
        fold_i_att_acc['Attributes'].append(nv)
        fold_i_att_acc['Train Acc'].append(zsl_accuracy_train)
        fold_i_att_acc['Val Acc'].append(zsl_accuracy1_unseen)
        fold_i_att_acc['Mask'].append(mask)
        
        if debug:
            print(f"nv: {nv} || Train: {zsl_accuracy_train} || Val: {zsl_accuracy1_unseen}")
    # Saving the best configuration of the fold
    fold_i_att_acc = pd.DataFrame(fold_i_att_acc).sort_values(by = 'Val Acc', ascending = False)
    best_folds_configuration.append(fold_i_att_acc.iloc[0])
    

for res in best_folds_configuration:
    conteo = conteo + np.asarray(res['Mask']).astype(int)
   
   
   
   
df = {}
for i in range(len(best_folds_configuration)):
    df[i] = best_folds_configuration[i].to_dict()
df = pd.DataFrame(df).T

if rfe_mode: # Los mejores resultados de cada fold y el conteo
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_rfe_val_val)
else:
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_random_val)
    
    
    
    
    
    
    
    
# Evaluamos en test 
test_results = {'Counts':[],'Attributes':[],'Train Acc':[],'Test Seen Acc':[],'Test Unseen Acc':[]}
j = 1
cadena = '5'
for i in range(FOLDS*2-1):
    if i < FOLDS:
        test_results['Counts'].append(str(FOLDS-i))
        test_results['Attributes'].append(np.where(conteo==FOLDS-i)[0].shape)
        mejores = np.where(conteo==FOLDS-i)[0]
        print(FOLDS-i)
    else:
        cadena = cadena+'-'+str(FOLDS-j)
        test_results['Counts'].append(cadena)
        test_results['Attributes'].append(np.where(conteo>=FOLDS-j)[0].shape)
        mejores = np.where(conteo>=FOLDS-j)[0]
        j+=1
        print(FOLDS-j)
    
    
    if mejores.shape[0] > 0:
        # Training
        training_feature = feature[trainval_loc].copy()
        training_label   = labels[trainval_loc].copy()
        ss_training      = gen_ss_from_data(training_label,attribute)
        # Get test
        test_unseen_feature = feature[test_unseen_loc].copy()
        test_unseen_label   = labels[test_unseen_loc].copy()
        ss_test_unseen      = gen_ss_from_data(test_unseen_label,attribute)
        test_seen_feature = feature[test_seen_loc].copy()
        test_seen_label   = labels[test_seen_loc].copy()
        ss_test_seen      = gen_ss_from_data(test_seen_label,attribute)
        # Preprocessing
        scaler = get_scaler(scaler_str)
        scaler.fit(training_feature)
        training_feature = scaler.transform(training_feature)
        test_unseen_feature = scaler.transform(test_unseen_feature)
        test_seen_feature = scaler.transform(test_seen_feature)
        # Fit the model
        W = SAE(training_feature.T, ss_training[:,mejores].T,500000) 
        gt_ss_training_adapted  = attribute[np.unique(training_label)][:,mejores]
        semantic_predicted = np.dot(training_feature, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, np.unique(training_label),training_label)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        test_results['Train Acc'].append(zsl_accuracy_train)
        # Evaluo en test seen
        gt_ss_val_adapted    = attribute[np.unique(test_seen_label)][:,mejores]
        semantic_predicted = np.dot(test_seen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_seen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_seen_label),test_seen_label)
        zsl_accuracy1_seen = round(zsl_accuracy1_seen,DECIMAL_NUMBERS)
        test_results['Test Seen Acc'].append(zsl_accuracy1_seen)
        # Evaluo en test unseen
        gt_ss_val_adapted    = attribute[np.unique(test_unseen_label)][:,mejores]
        semantic_predicted = np.dot(test_unseen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_unseen_label),test_unseen_label)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        test_results['Test Unseen Acc'].append(zsl_accuracy1_unseen)
    else:
        test_results['Train Acc'].append(0)
        test_results['Test Seen Acc'].append(0)
        test_results['Test Unseen Acc'].append(0)
    
if rfe_mode:
    pd.DataFrame(test_results).to_csv(path_oracle_rfe_val_val)
else:
    pd.DataFrame(test_results).to_csv(path_oracle_random_val)
    
pd.DataFrame(test_results)

You've selected SUN dataset
You've selected rfe mode
Fold 1...
Ranking: [58, 29, 64, 84, 98, 5, 53, 55, 90, 62, 52, 32, 37, 7, 59, 31, 19, 46, 47, 44, 92, 45, 82, 50, 28, 39, 23, 16, 65, 36, 21, 41, 35, 11, 26, 10, 22, 70, 77, 51, 34, 60, 85, 95, 88, 1, 54, 73, 20, 61, 78, 27, 8, 93, 80, 97, 101, 18, 74, 66, 40, 13, 15, 76, 33, 83, 72, 43, 4, 71, 48, 3, 75, 68, 24, 100, 0, 99, 56, 2, 42, 30, 87, 17, 89, 38, 94, 63, 12, 91, 9, 57, 69, 25, 67, 81, 14, 49, 96, 6, 86, 79]
nv: 10 || Train: 20.797 || Val: 13.517
nv: 15 || Train: 32.485 || Val: 18.362
nv: 20 || Train: 40.879 || Val: 21.56
nv: 25 || Train: 44.985 || Val: 24.322
nv: 30 || Train: 51.017 || Val: 27.519
nv: 35 || Train: 55.584 || Val: 30.281
nv: 40 || Train: 58.854 || Val: 31.783
nv: 45 || Train: 61.337 || Val: 32.558
nv: 50 || Train: 62.379 || Val: 33.672
nv: 55 || Train: 64.62 || Val: 34.302
nv: 60 || Train: 64.668 || Val: 34.109
nv: 65 || Train: 65.346 || Val: 34.593
nv: 70 || Train: 66.255 || Val: 35.32
nv: 75 || Train: 67.151

,Counts,Attributes,Train Acc,Test Seen Acc,Test Unseen Acc
0,5,"(101,)",61.240,28.953,46.736
1,4,"(1,)",0.223,0.078,2.292
2,3,"(0,)",0.000,0.000,0.000
3,2,"(0,)",0.000,0.000,0.000
4,1,"(0,)",0.000,0.000,0.000
5,5-4,"(102,)",61.531,29.419,47.153
6,5-4-3,"(102,)",61.531,29.419,47.153
7,5-4-3-2,"(102,)",61.531,29.419,47.153
8,5-4-3-2-1,"(102,)",61.531,29.419,47.153


# Animals

In [7]:
#################
### Arguments ###
#################
class Arguments:
    def __init__(self, dataset,seed,mode):
        self.dataset = dataset
        self.seed = seed
        self.mode = mode
        
         
DECIMAL_NUMBERS        = 3
FOLDS                  = 5
scaler_str             = 'Standard'     # ['Standard','MinMax']
orig_attribute         = False # Attributos originales u otros
debug                  = True 
SHOW                   = True

step                     = 5  # Paso de atributos a seleccionar [a, a+step, a+2*step, ...]
hitk = 1    
args = Arguments('AWA2',42,'rfe') # MODE = RFE


#################
##### SetUp #####
#################
random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True

# Set up the path to the data
# Set up the path to the data
path_dir                  = pathlib.Path('data/')
path_data                 = path_dir / args.dataset  
results_dir               = pathlib.Path(f'Results/HAIS_v2/{args.dataset}') 


path_results_rfe_train_val    = results_dir / f'Wrapper{args.dataset}-RFE-train-val.csv'
path_results_random_val       = results_dir / f'Wrapper{args.dataset}-Random-val.csv' 
path_results_rfe_val_val      = results_dir / f'Wrapper{args.dataset}-RFE-val-val.csv'
path_results_random_simple    = results_dir / f'Wrapper{args.dataset}-Random-simple.csv' 

path_masks_rfe_train_val    = results_dir / f'Masks_{args.dataset}-RFE-train-val.csv'
path_masks_random_val       = results_dir / f'Masks_{args.dataset}-Random-val.csv' 
path_masks_rfe_val_val      = results_dir / f'Masks_{args.dataset}-RFE-val-val.csv'
path_masks_random_simple    = results_dir / f'Masks_{args.dataset}-Random-simple.csv' 

path_oracle_rfe_train_val    = results_dir / f'Oracle_Wrapper_{args.dataset}-RFE-train-val.csv'
path_oracle_random_val       = results_dir / f'Oracle_Wrapper_{args.dataset}-Random-val.csv' 
path_oracle_rfe_val_val      = results_dir / f'Oracle_Wrapper_{args.dataset}-RFE-val-val.csv'
path_oracle_random_simple    = results_dir / f'Oracle_Wrapper_{args.dataset}-Random-simple.csv' 

path_conteo_rfe_train_val    = results_dir / f'Conteo_{args.dataset}-RFE-train-val.csv'
path_conteo_random_val       = results_dir / f'Conteo_{args.dataset}-Random-val.csv' 
path_conteo_rfe_val_val      = results_dir / f'Conteo_{args.dataset}-RFE-val-val.csv'
path_conteo_random_simple    = results_dir / f'Conteo_{args.dataset}-Random-simple.csv' 



# Load the data
matcontent, att_splits = load_data_from_path(path_data, show = False)

"""
Comentar que en los datos de este datasets viene una partición establecida para train y val. Sin embargo, la partición de 'val' no contempla clases Unseen. Por tanto no la vamos usar.
Crearemos una partición de validación para clases Seen y Unseen a partir de la partición de test. 
Como conjunto de entrenamiento usaremos el que viene dado por los índices de la variable 'trainval_loc' en att_splits. 
Los índices de las varaibles 'train_loc' y 'val_loc' no los vamos a usar.
"""
# Get the data from att_splits dictionary
allclasses_names            = att_splits['allclasses_names']
original_att                = att_splits['original_att'].T.astype('float')  # Attributos originales
attribute                   = att_splits['att'].T.astype('float')           # Atributos que no se de donde salen, pero son los que usan <=== IMPORTANTE
# Indixes of the instances in the train, test and validation sets
test_unseen_loc             = att_splits['test_unseen_loc'].squeeze() - 1
test_seen_loc               = att_splits['test_seen_loc'].squeeze() - 1
trainval_loc                = att_splits['trainval_loc'].squeeze() - 1      # En caso de no usar validación, aquí estara todo el entrenamiento
train_loc                   = att_splits['train_loc'].squeeze() - 1         # Esto es si usamos validacion, ver comentario arriba
val_loc                     = att_splits['val_loc'].squeeze() -1            # Esto es si usamos validacion, ver comentario arriba
# Get the dta from matcontent dictionary
feature =  matcontent['features'].T                       # Feature Matrix (Number of instances x Number of features)
labels  =  matcontent['labels'].astype(int).squeeze() - 1 # Ponemos que sea entero, eliminamos la dimension extra y con el '-1' movemos el rango de [1,200] a [0,199]

random.shuffle(trainval_loc)
# Es importante usar aquí la matrices feature y labels originales
trainval_feature = feature[trainval_loc]
trainval_labels  = labels[trainval_loc]
trainval_classes = np.unique(trainval_labels)
random.shuffle(trainval_classes)

classes_per_fold = int(trainval_classes.shape[0] / FOLDS)
acarreo          = trainval_classes.shape[0] % FOLDS


# Get attributes and preprocessing them
attribute = original_att if orig_attribute else attribute
attribute = scaler_data(get_scaler(scaler_str),attribute)




#################
##### FOLDS #####
#################
folds = []
for i in range(FOLDS):
    
    # Gather data for the fold
    start = i * classes_per_fold
    end   = (i + 1) * classes_per_fold if i != FOLDS - 1 else (i + 1) * classes_per_fold + acarreo
    
    fold_i_val_classes       = trainval_classes[start:end] # Unseen
    fold_i_training_classes  = [x for x in trainval_classes if x not in fold_i_val_classes] # Seen
    
    fold_i_val_loc           = np.where(np.isin(trainval_labels, fold_i_val_classes)) # index for val
    fold_i_training_loc      = np.where(np.isin(trainval_labels, fold_i_training_classes)) # Index for training
    # np.isin(element,test_element) Calculates element in test_elements, broadcasting over element only. 
    # #Returns a boolean array of the same shape as element that is True where an element of element is in 
    # #test_elements and False otherwise.
    # np.where(lists[bool]) returns the indexes of the True values in the lists
    
    assert np.intersect1d(fold_i_val_loc, fold_i_training_loc).shape[0] == 0, f"Error: Problems found in creating folds {i}. Val and training classes are not disjoint"
    assert np.intersect1d(fold_i_val_classes, fold_i_training_classes).shape[0] == 0, f"Error: Problems found in creating folds {i}. Val and training classes are not disjoint"
    assert all([i not in fold_i_training_classes for i in trainval_labels[fold_i_val_loc]]), f"Error: Problems found in creating folds {i}"
    assert all([i not in fold_i_val_classes for i in trainval_labels[fold_i_training_loc]]), f"Error: Problems found in creating folds {i}"

    fold_i_training_features = trainval_feature[fold_i_training_loc]
    fold_i_val_features      = trainval_feature[fold_i_val_loc]
    fold_i_training_labels   = trainval_labels[fold_i_training_loc]
    fold_i_val_labels        = trainval_labels[fold_i_val_loc]
    fold_i_ss_val            = gen_ss_from_data(fold_i_val_labels,attribute)
    fold_i_ss_training       = gen_ss_from_data(fold_i_training_labels,attribute)
    
    assert np.intersect1d(np.unique(fold_i_val_labels),np.unique(fold_i_training_labels)).shape[0] == 0, f"Los conjuntos del fold {i+1} de etiquetas no son disjuntos"
    
    # Preprocess the data
    fold_i_training_features, fold_i_val_features = scaler_data(get_scaler(scaler_str),
                                                                fold_i_training_features,
                                                                fold_i_val_features)
    
    # Creating Datasets 
    fold_training = Dataset(features = fold_i_training_features, 
                            labels   = fold_i_training_labels, 
                            att      = fold_i_ss_training, 
                            mode     = 'train')
    
    fold_val      = Dataset(features = fold_i_val_features,
                            labels   = fold_i_val_labels,
                            att      = fold_i_ss_val,
                            mode     = 'val')
    
    fold = StackedDataset({'train':fold_training, 'val':fold_val})
    
    folds.append(fold)
    
    if SHOW:
        print(f"Fold {i+1} || Val Classes {np.unique(fold_i_val_labels).shape} - {fold_i_val_labels.shape} instances || Training Classes {np.unique(fold_i_training_labels).shape} - {fold_i_training_labels.shape} instances")
        
        

You've selected AWA2 dataset
You've selected rfe mode
Fold 1 || Val Classes (8,) - (4631,) instances || Training Classes (32,) - (18896,) instances
Fold 2 || Val Classes (8,) - (5050,) instances || Training Classes (32,) - (18477,) instances
Fold 3 || Val Classes (8,) - (4616,) instances || Training Classes (32,) - (18911,) instances
Fold 4 || Val Classes (8,) - (4629,) instances || Training Classes (32,) - (18898,) instances
Fold 5 || Val Classes (8,) - (4601,) instances || Training Classes (32,) - (18926,) instances


## Train VAl

In [8]:
args = Arguments('AWA2',42,'rfe') # MODE = RFE
random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True

vattributes = np.arange(10, attribute.shape[1], step)
if attribute.shape[1] % step != 0:
    vattributes = np.append(vattributes, attribute.shape[1])

conteo          = np.zeros(attribute.shape[1])
best_folds_configuration = [] # Mejor configurración de atributos seleccionados en cada uno de los folds
for i, fold in enumerate(folds):

    print(f"Fold {i+1}...")

    training_data = fold.datasets['train']
    val_data      = fold.datasets['val']

    attributes_X_train = training_data.att.copy()

    rfe = RFE(estimator = None, estimator_name = 'SAE', n_features_to_select = 1, step = 1, verbose = True)
    rfe.fit(training_data.features, training_data.att,training_data.labels)
    ranking = rfe.get_ranking()
    
    fold_i_att_acc = {'Attributes': [],
                      'Train Acc': [],
                      'Val Acc': [],
                      'Mask':[]}
    
    for nv in vattributes:
        
        mask = ranking <= nv
        W = SAE(training_data.features.T, training_data.att[:,mask].T,500000) 
        # Training Acc
        gt_ss_training_adapted  = attribute[training_data.classes][:,mask]
        semantic_predicted = np.dot(training_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, training_data.classes,training_data.labels)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        # Val Acc
        gt_ss_val_adapted    = attribute[val_data.classes][:,mask]
        semantic_predicted = np.dot(val_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, val_data.classes,val_data.labels)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        # Save results
        fold_i_att_acc['Attributes'].append(nv)
        fold_i_att_acc['Train Acc'].append(zsl_accuracy_train)
        fold_i_att_acc['Val Acc'].append(zsl_accuracy1_unseen)
        fold_i_att_acc['Mask'].append(mask)
        
        if debug:
            print(f"nv: {nv} || Train: {zsl_accuracy_train} || Val: {zsl_accuracy1_unseen}")
    # Saving the best configuration of the fold
    fold_i_att_acc = pd.DataFrame(fold_i_att_acc).sort_values(by = 'Val Acc', ascending = False)
    best_folds_configuration.append(fold_i_att_acc.iloc[0])
    

for res in best_folds_configuration:
    conteo = conteo + np.asarray(res['Mask']).astype(int)
    
    
df = {}
for i in range(len(best_folds_configuration)):
    df[i] = best_folds_configuration[i].to_dict()
df = pd.DataFrame(df).T

if rfe_mode: 
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_rfe_train_val)
else:
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_random_val)
    
    
    
    
# Evaluamos en test 
test_results = {'Counts':[],'Attributes':[],'Train Acc':[],'Test Seen Acc':[],'Test Unseen Acc':[]}
j = 1
cadena = '5'
for i in range(FOLDS*2-1):
    if i < FOLDS:
        test_results['Counts'].append(str(FOLDS-i))
        test_results['Attributes'].append(np.where(conteo==FOLDS-i)[0].shape)
        mejores = np.where(conteo==FOLDS-i)[0]

    else:
        cadena = cadena+'-'+str(FOLDS-j)
        test_results['Counts'].append(cadena)
        test_results['Attributes'].append(np.where(conteo>=FOLDS-j)[0].shape)
        mejores = np.where(conteo>=FOLDS-j)[0]
        j+=1

    
    
    if mejores.shape[0] > 0:
        # Training
        training_feature = feature[trainval_loc]
        training_label   = labels[trainval_loc]
        ss_training      = gen_ss_from_data(training_label,attribute)
        # Get test
        test_unseen_feature = feature[test_unseen_loc]
        test_unseen_label   = labels[test_unseen_loc]
        ss_test_unseen      = gen_ss_from_data(test_unseen_label,attribute)
        test_seen_feature = feature[test_seen_loc]
        test_seen_label   = labels[test_seen_loc]
        ss_test_seen      = gen_ss_from_data(test_seen_label,attribute)
        # Preprocessing
        scaler = get_scaler(scaler_str)
        scaler.fit(training_feature)
        training_feature = scaler.transform(training_feature)
        test_unseen_feature = scaler.transform(test_unseen_feature)
        test_seen_feature = scaler.transform(test_seen_feature)
        # Fit the model
        W = SAE(training_feature.T, ss_training[:,mejores].T,500000) 
        gt_ss_training_adapted  = attribute[np.unique(training_label)][:,mejores]
        semantic_predicted = np.dot(training_feature, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, np.unique(training_label),training_label)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        test_results['Train Acc'].append(zsl_accuracy_train)
        # Evaluo en test seen
        gt_ss_val_adapted    = attribute[np.unique(test_seen_label)][:,mejores]
        semantic_predicted = np.dot(test_seen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_seen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_seen_label),test_seen_label)
        zsl_accuracy1_seen = round(zsl_accuracy1_seen,DECIMAL_NUMBERS)
        test_results['Test Seen Acc'].append(zsl_accuracy1_seen)
        # Evaluo en test unseen
        gt_ss_val_adapted    = attribute[np.unique(test_unseen_label)][:,mejores]
        semantic_predicted = np.dot(test_unseen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_unseen_label),test_unseen_label)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        test_results['Test Unseen Acc'].append(zsl_accuracy1_unseen)
    else:
        test_results['Train Acc'].append(0)
        test_results['Test Seen Acc'].append(0)
        test_results['Test Unseen Acc'].append(0)
    
if rfe_mode:
    pd.DataFrame(test_results).to_csv(path_oracle_rfe_train_val)
else:
    pd.DataFrame(test_results).to_csv(path_oracle_random_val)
    
pd.DataFrame(test_results)

You've selected AWA2 dataset
You've selected rfe mode
Fold 1...
Ranking: [30, 6, 3, 32, 66, 24, 80, 41, 43, 20, 5, 52, 4, 83, 62, 42, 17, 33, 38, 12, 75, 11, 79, 55, 34, 72, 51, 27, 48, 61, 8, 23, 58, 36, 0, 76, 40, 78, 26, 7, 59, 15, 81, 28, 65, 54, 77, 21, 67, 22, 69, 53, 2, 9, 60, 31, 35, 82, 47, 84, 45, 49, 19, 44, 18, 50, 16, 29, 74, 64, 10, 14, 46, 39, 71, 1, 56, 13, 70, 57, 63, 68, 25, 37, 73]
nv: 10 || Train: 88.172 || Val: 54.697
nv: 15 || Train: 92.692 || Val: 57.871
nv: 20 || Train: 94.74 || Val: 55.042
nv: 25 || Train: 95.258 || Val: 62.773
nv: 30 || Train: 95.57 || Val: 65.385
nv: 35 || Train: 96.084 || Val: 72.403
nv: 40 || Train: 96.041 || Val: 76.571
nv: 45 || Train: 96.031 || Val: 76.247
nv: 50 || Train: 96.206 || Val: 79.162
nv: 55 || Train: 96.274 || Val: 81.214
nv: 60 || Train: 96.306 || Val: 81.689
nv: 65 || Train: 96.417 || Val: 82.531
nv: 70 || Train: 96.47 || Val: 78.493
nv: 75 || Train: 96.507 || Val: 79.292
nv: 80 || Train: 96.565 || Val: 85.727
Fold 2...
Rank

,Counts,Attributes,Train Acc,Test Seen Acc,Test Unseen Acc
0,5,"(11,)",90.653,86.348,42.260
1,4,"(49,)",95.325,92.095,38.885
2,3,"(24,)",93.582,89.918,26.855
3,2,"(1,)",4.259,3.808,16.340
4,1,"(0,)",0.000,0.000,0.000
5,5-4,"(60,)",95.575,92.333,44.547
6,5-4-3,"(84,)",95.856,92.894,39.189
7,5-4-3-2,"(85,)",95.856,92.860,39.151
8,5-4-3-2-1,"(85,)",95.856,92.860,39.151


## Val Val

In [9]:
args = Arguments('AWA2',42,'rfe') # MODE = RFE


random.seed(args.seed)
np.random.seed(args.seed)
print(f"You've selected {args.dataset} dataset")
print(f"You've selected {args.mode} mode")
random_mode = False if args.mode == 'rfe' else True 
rfe_mode    = False if args.mode == 'random' else True




vattributes = np.arange(10, attribute.shape[1], step)
if attribute.shape[1] % step != 0:
    vattributes = np.append(vattributes, attribute.shape[1])

conteo          = np.zeros(attribute.shape[1])
best_folds_configuration = [] # Mejor configurración de atributos seleccionados en cada uno de los folds
for i, fold in enumerate(folds):

    print(f"Fold {i+1}...")

    training_data = fold.datasets['train']
    val_data      = fold.datasets['val']



    rfe = RFE(estimator = None, estimator_name = 'SAE', n_features_to_select = 1, step = 1, verbose = True)
    rfe.fit(val_data.features, val_data.att,val_data.labels)
    ranking = rfe.get_ranking()
    
    fold_i_att_acc = {'Attributes': [],
                      'Train Acc': [],
                      'Val Acc': [],
                      'Mask':[]}
    
    for nv in vattributes:
        
        mask = ranking <= nv
        W = SAE(training_data.features.T, training_data.att[:,mask].T,500000) 
        # Training Acc
        gt_ss_training_adapted  = attribute[training_data.classes][:,mask]
        semantic_predicted = np.dot(training_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, training_data.classes,training_data.labels)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        # Val Acc
        gt_ss_val_adapted    = attribute[val_data.classes][:,mask]
        semantic_predicted = np.dot(val_data.features, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, val_data.classes,val_data.labels)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        # Save results
        fold_i_att_acc['Attributes'].append(nv)
        fold_i_att_acc['Train Acc'].append(zsl_accuracy_train)
        fold_i_att_acc['Val Acc'].append(zsl_accuracy1_unseen)
        fold_i_att_acc['Mask'].append(mask)
        
        if debug:
            print(f"nv: {nv} || Train: {zsl_accuracy_train} || Val: {zsl_accuracy1_unseen}")
    # Saving the best configuration of the fold
    fold_i_att_acc = pd.DataFrame(fold_i_att_acc).sort_values(by = 'Val Acc', ascending = False)
    best_folds_configuration.append(fold_i_att_acc.iloc[0])
    

for res in best_folds_configuration:
    conteo = conteo + np.asarray(res['Mask']).astype(int)
   
   
   
   
df = {}
for i in range(len(best_folds_configuration)):
    df[i] = best_folds_configuration[i].to_dict()
df = pd.DataFrame(df).T

if rfe_mode: # Los mejores resultados de cada fold y el conteo
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_rfe_val_val)
else:
    df[['Attributes','Train Acc', 'Val Acc','Mask']].to_csv(path_results_random_val)
    
    
    
    
    
    
    
    
# Evaluamos en test 
test_results = {'Counts':[],'Attributes':[],'Train Acc':[],'Test Seen Acc':[],'Test Unseen Acc':[]}
j = 1
cadena = '5'
for i in range(FOLDS*2-1):
    if i < FOLDS:
        test_results['Counts'].append(str(FOLDS-i))
        test_results['Attributes'].append(np.where(conteo==FOLDS-i)[0].shape)
        mejores = np.where(conteo==FOLDS-i)[0]
        print(FOLDS-i)
    else:
        cadena = cadena+'-'+str(FOLDS-j)
        test_results['Counts'].append(cadena)
        test_results['Attributes'].append(np.where(conteo>=FOLDS-j)[0].shape)
        mejores = np.where(conteo>=FOLDS-j)[0]
        j+=1
        print(FOLDS-j)
    
    
    if mejores.shape[0] > 0:
        # Training
        training_feature = feature[trainval_loc].copy()
        training_label   = labels[trainval_loc].copy()
        ss_training      = gen_ss_from_data(training_label,attribute)
        # Get test
        test_unseen_feature = feature[test_unseen_loc].copy()
        test_unseen_label   = labels[test_unseen_loc].copy()
        ss_test_unseen      = gen_ss_from_data(test_unseen_label,attribute)
        test_seen_feature = feature[test_seen_loc].copy()
        test_seen_label   = labels[test_seen_loc].copy()
        ss_test_seen      = gen_ss_from_data(test_seen_label,attribute)
        # Preprocessing
        scaler = get_scaler(scaler_str)
        scaler.fit(training_feature)
        training_feature = scaler.transform(training_feature)
        test_unseen_feature = scaler.transform(test_unseen_feature)
        test_seen_feature = scaler.transform(test_seen_feature)
        # Fit the model
        W = SAE(training_feature.T, ss_training[:,mejores].T,500000) 
        gt_ss_training_adapted  = attribute[np.unique(training_label)][:,mejores]
        semantic_predicted = np.dot(training_feature, normalizeFeature(W).transpose())
        [zsl_accuracy_train, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_training_adapted, hitk, np.unique(training_label),training_label)
        zsl_accuracy_train = round(zsl_accuracy_train,DECIMAL_NUMBERS)
        test_results['Train Acc'].append(zsl_accuracy_train)
        # Evaluo en test seen
        gt_ss_val_adapted    = attribute[np.unique(test_seen_label)][:,mejores]
        semantic_predicted = np.dot(test_seen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_seen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_seen_label),test_seen_label)
        zsl_accuracy1_seen = round(zsl_accuracy1_seen,DECIMAL_NUMBERS)
        test_results['Test Seen Acc'].append(zsl_accuracy1_seen)
        # Evaluo en test unseen
        gt_ss_val_adapted    = attribute[np.unique(test_unseen_label)][:,mejores]
        semantic_predicted = np.dot(test_unseen_feature, normalizeFeature(W).transpose())
        [zsl_accuracy1_unseen, y_hit_k] = zsl_acc(semantic_predicted, gt_ss_val_adapted, hitk, np.unique(test_unseen_label),test_unseen_label)
        zsl_accuracy1_unseen = round(zsl_accuracy1_unseen,DECIMAL_NUMBERS)
        test_results['Test Unseen Acc'].append(zsl_accuracy1_unseen)
    else:
        test_results['Train Acc'].append(0)
        test_results['Test Seen Acc'].append(0)
        test_results['Test Unseen Acc'].append(0)
    
if rfe_mode:
    pd.DataFrame(test_results).to_csv(path_oracle_rfe_val_val)
else:
    pd.DataFrame(test_results).to_csv(path_oracle_random_val)
    
pd.DataFrame(test_results)

You've selected AWA2 dataset
You've selected rfe mode
Fold 1...
Ranking: [10, 35, 63, 74, 1, 67, 40, 13, 62, 0, 12, 75, 42, 36, 39, 43, 3, 5, 26, 30, 51, 83, 73, 18, 64, 65, 11, 16, 14, 41, 47, 9, 57, 82, 46, 54, 29, 71, 28, 81, 53, 60, 4, 32, 80, 68, 45, 8, 15, 25, 38, 34, 55, 66, 33, 79, 2, 72, 84, 61, 56, 22, 49, 58, 48, 7, 20, 50, 27, 31, 6, 19, 78, 59, 77, 24, 44, 69, 70, 23, 17, 52, 37, 21, 76]
nv: 10 || Train: 90.543 || Val: 34.55
nv: 15 || Train: 92.448 || Val: 47.247
nv: 20 || Train: 94.242 || Val: 69.1
nv: 25 || Train: 94.761 || Val: 67.221
nv: 30 || Train: 94.692 || Val: 70.244
nv: 35 || Train: 95.232 || Val: 74.476
nv: 40 || Train: 95.533 || Val: 69.445
nv: 45 || Train: 95.518 || Val: 68.646
nv: 50 || Train: 95.899 || Val: 68.387
nv: 55 || Train: 96.036 || Val: 80.76
nv: 60 || Train: 96.068 || Val: 81.926
nv: 65 || Train: 96.258 || Val: 83.783
nv: 70 || Train: 96.296 || Val: 84.517
nv: 75 || Train: 96.428 || Val: 85.338
nv: 80 || Train: 96.507 || Val: 85.943
Fold 2...
Ranki

,Counts,Attributes,Train Acc,Test Seen Acc,Test Unseen Acc
0,5,"(21,)",92.341,88.388,49.703
1,4,"(33,)",94.143,90.785,29.433
2,3,"(26,)",93.080,90.020,33.982
3,2,"(5,)",60.195,55.423,38.532
4,1,"(0,)",0.000,0.000,0.000
5,5-4,"(54,)",95.316,91.942,34.652
6,5-4-3,"(80,)",95.886,92.860,38.140
7,5-4-3-2,"(85,)",95.856,92.860,39.151
8,5-4-3-2-1,"(85,)",95.856,92.860,39.151
